# 20_04_勾配の求め方と_つまずきやすい_3_点

コード：`ch20/blocks/20_04_勾配の求め方と_つまずきやすい_3_点.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 付録 A-4：自動微分。計算の記録から、勾配を後ろ向きに求める
import torch

w = torch.tensor(3.0, requires_grad=True)                 # 勾配が欲しい値に印を付ける
x = torch.tensor(2.0)
y = (w * x - 1) ** 2                                      # y = (wx − 1)² = 25
y.backward()                                              # dy/dw = 2(wx − 1)·x = 2·5·2 = 20
print("y", y.item(), " dy/dw", w.grad.item())

y = (w * x - 1) ** 2
y.backward()                                              # ★ もう一度 backward すると、勾配は足し込まれる
print("2 回目の backward のあと", w.grad.item(), "（20 + 20）")
w.grad = None                                             # 学習ループで zero_grad が要る理由
y = (w * x - 1) ** 2; y.backward()
print("消してから", w.grad.item())

with torch.no_grad():                                     # 評価では記録しない（メモリと時間の節約）
    y2 = (w * x - 1) ** 2
print("no_grad の中の計算は記録されるか", y2.requires_grad)
d = (w * 2).detach()                                      # detach：ここから先を記録から切り離す
print("detach したものは記録されるか", d.requires_grad)

# 4 章 06 節の手計算と同じネットを、自動微分で
W1 = torch.tensor([[0.1, -0.2], [0.4, 0.3]], requires_grad=True)
b1 = torch.tensor([0.0, 0.1], requires_grad=True)
w2 = torch.tensor([0.5, -0.6], requires_grad=True); b2 = torch.tensor(0.2, requires_grad=True)
L = 0.5 * (w2 @ torch.relu(W1 @ torch.tensor([1.0, 2.0]) + b1) + b2 - 1.0) ** 2
L.backward()
print("L", round(L.item(), 4), " ∂L/∂w2", [round(v, 3) for v in w2.grad.tolist()], " ∂L/∂W1 の 2 行目", [round(v, 3) for v in W1.grad[1].tolist()])
